In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set plotting style
sns.set_theme(style="whitegrid", context="notebook")

class ExperimentVisualizer:
    """
    A class to load, process, and visualize Federated Learning experiment data.
    """

    def __init__(self, experiment_path: str):
        self.path = Path(experiment_path)
        self.metadata = None
        self.server_df = None
        self.client_history_df = None
        self.client_eval_df = None
        
        print(f"Loading data from: {self.path}")
        self._load_data()

    def _load_data(self):
        """Loads all available CSV and JSON files from the experiment folder."""
        
        # 1. Load Metadata
        meta_path = self.path / "metadata.json"
        if meta_path.exists():
            with open(meta_path, "r") as f:
                self.metadata = json.load(f)
            print("  [✓] Loaded metadata.json")
        else:
            print("  [!] metadata.json not found.")

        # 2. Load Server Metrics
        server_path = self.path / "server_metrics.csv"
        if server_path.exists():
            self.server_df = pd.read_csv(server_path)
            print("  [✓] Loaded server_metrics.csv")
        else:
            print("  [!] server_metrics.csv not found.")

        # 3. Load Client History
        hist_path = self.path / "client_history.csv"
        if hist_path.exists():
            self.client_history_df = pd.read_csv(hist_path)
            print("  [✓] Loaded client_history.csv")
        else:
            print("  [!] client_history.csv not found.")

        # 4. Load Client Evaluation
        eval_path = self.path / "client_evaluation.csv"
        if eval_path.exists():
            self.client_eval_df = pd.read_csv(eval_path)
            print("  [✓] Loaded client_evaluation.csv")
        else:
            print("  [!] client_evaluation.csv not found.")

    def plot_server_metrics(self):
        """
        Plots Server Accuracy and Loss over rounds in a single figure.
        """
        if self.server_df is None:
            print("No server data to plot.")
            return

        fig, ax1 = plt.subplots(figsize=(10, 5))

        # Plot Accuracy
        color = 'tab:blue'
        ax1.set_xlabel('Round')
        ax1.set_ylabel('Global Accuracy', color=color)
        sns.lineplot(data=self.server_df, x='round', y='accuracy', ax=ax1, color=color, marker='o', label='Global Accuracy')
        ax1.tick_params(axis='y', labelcolor=color)

        # Plot Loss on secondary axis
        ax2 = ax1.twinx()
        color = 'tab:red'
        ax2.set_ylabel('Global Loss', color=color)
        sns.lineplot(data=self.server_df, x='round', y='loss', ax=ax2, color=color, marker='x', linestyle='--', label='Global Loss')
        ax2.tick_params(axis='y', labelcolor=color)

        # Fix Legend: Combine handles, place outside, make opaque
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper center', 
                   bbox_to_anchor=(0.5, -0.15), ncol=2, framealpha=1.0)
        
        # Remove the second legend created by seaborn automatically
        if ax2.get_legend():
            ax2.get_legend().remove()

        plt.title('Server Performance Over Rounds')
        plt.tight_layout()
        plt.show()

    def plot_client_vs_server(self):
        """
        Plots Client Validation Accuracy vs Server Global Accuracy.
        """
        if self.client_eval_df is None or self.server_df is None:
            print("Missing client or server data for comparison.")
            return

        fig, ax = plt.subplots(figsize=(12, 6))

        # Plot Server Global Accuracy (Thicker line for emphasis)
        sns.lineplot(data=self.server_df, x='round', y='accuracy', 
                     color='black', linewidth=2.5, marker='s', label='Server (Global)', zorder=10, ax=ax)

        # Plot Client Accuracies
        sns.lineplot(data=self.client_eval_df, x='round', y='accuracy', 
                     hue='client-name', style='client-name', markers=True, dashes=False, ax=ax)
        
        ax.set_title('Client Local vs Server Global Accuracy')
        ax.set_ylabel('Accuracy')
        
        # Fix Legend: Place outside to the right, make opaque
        ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), framealpha=1.0, borderaxespad=0.0)

        plt.tight_layout()
        plt.show()

    def plot_system_performance(self):
        """
        Plots Round Time, Transmission Time, and Client Train Time.
        """
        if self.client_eval_df is None:
            return

        fig, ax = plt.subplots(figsize=(12, 6))
        
        # FIX: Add numeric_only=True to ignore 'client-name' strings
        agg_client = self.client_eval_df.groupby('round').mean(numeric_only=True).reset_index()

        # Bar width for grouped bars
        width = 0.35
        x = agg_client['round']
        
        # Plot Average Client Train Time
        # Note: We assign the output to 'rects1' to use for the legend later
        rects1 = ax.bar(x - width/2, agg_client['train-time'], width, label='Avg Client Train Time', color='skyblue')
        
        # Plot Average Transmission Time (Overhead)
        rects2 = ax.bar(x + width/2, agg_client['transmission-time'], width, label='Avg Transmission/Wait Time', color='salmon')

        ax.set_ylabel('Time (seconds)')
        ax.set_xlabel('Round')
        ax.set_title('System Performance: Training vs Transmission Overhead')
        
        # --- Handle Legends ---
        if self.server_df is not None and 'round-time' in self.server_df.columns:
            # Create secondary axis
            ax2 = ax.twinx()
            
            # Plot line (Seaborn adds its own legend automatically, so we capture it)
            sns.lineplot(data=self.server_df, x='round', y='round-time', color='green', marker='o', ax=ax2, label='Total Round Time')
            ax2.set_ylabel('Total Round Time (s)', color='green')
            ax2.tick_params(axis='y', labelcolor='green')

            # 1. Remove the automatic legend created by Seaborn on ax2
            if ax2.get_legend():
                ax2.get_legend().remove()

            # 2. Combine handles and labels from both axes manually
            handles1, labels1 = ax.get_legend_handles_labels() # From bars
            handles2, labels2 = ax2.get_legend_handles_labels() # From line
            
            # 3. Create a single unified legend on ax (or fig)
            # Place it in the upper left to avoid the right axis label
            ax.legend(handles1 + handles2, labels1 + labels2, loc='upper left', framealpha=1.0)
            
        else:
            # Fallback if no server data: just show bar legend
            ax.legend(framealpha=1.0)

        plt.tight_layout()
        plt.show()

    def plot_global_confusion_matrix(self):
        """
        Plots the Global Confusion Matrix stored in metadata.
        """
        if not self.metadata or 'final_global_metrics' not in self.metadata:
            print("Metadata not found.")
            return

        # Extract confusion matrix list
        cm_list = self.metadata['final_global_metrics'].get('confusion-matrix')
        
        if not cm_list:
            print("Confusion matrix not found in metadata.")
            return

        # Convert list to numpy array
        cm_array = np.array(cm_list)
        
        # Normalize the confusion matrix
        row_sums = cm_array.sum(axis=1)
        cm_normalized = cm_array / row_sums[:, np.newaxis]
        
        # Replace NaN with 0
        cm_normalized = np.nan_to_num(cm_normalized)

        plt.figure(figsize=(10, 8))
        sns.heatmap(cm_normalized, annot=True, fmt=".2f", cmap="Blues", 
                    cbar=True, square=True)
        
        plt.title('Global Confusion Matrix (Normalized)')
        plt.xlabel('Predicted Label')
        plt.ylabel('True Label')
        plt.tight_layout()
        plt.show()

    def plot_straggler_analysis(self):
        """
        Visualizes stragglers by plotting the distribution of Client Train Times 
        vs the Total Round Duration.
        """
        if self.client_eval_df is None or self.server_df is None:
            print("Missing data for straggler analysis.")
            return

        fig, ax = plt.subplots(figsize=(12, 6))

        # 1. Box Plot of Client Train Times per Round
        # This shows the distribution: fast clients are low, stragglers are high dots
        sns.boxplot(data=self.client_eval_df, x='round', y='train-time', 
                    color='lightblue', width=0.4, ax=ax, fliersize=5, 
                    flierprops={'marker': 'o', 'markerfacecolor': 'red', 'alpha': 0.5})

        # 2. Overlay Line for Total Round Duration
        # We need to map round-time (indexed by round) to the x-axis (categorical or numerical)
        # Seaborn boxplot uses categorical x-axis (0, 1, 2...), so we can plot directly.
        
        # Extract unique rounds sorted
        rounds = sorted(self.client_eval_df['round'].unique())
        
        # Get corresponding round times from server_df
        round_times = [self.server_df.loc[self.server_df['round'] == r, 'round-time'].values[0] 
                       for r in rounds]
        
        # Plot the round duration line
        sns.lineplot(x=range(len(rounds)), y=round_times, color='black', 
                     marker='o', linewidth=2, label='Total Round Duration', ax=ax)

        ax.set_title('Straggler Analysis: Client Train Time vs Round Duration')
        ax.set_ylabel('Time (seconds)')
        ax.set_xlabel('Round')
        
        # Highlight the gap: The space between the top whisker/points and the black line 
        # is the "Wait Time" caused by stragglers.
        ax.legend(loc='upper right', framealpha=1.0)
        plt.tight_layout()
        plt.show()

    def plot_client_metrics_grid(self):
        """
        Plots Accuracy, Precision, Recall, and F1 for all clients in a 2x2 grid.
        """
        if self.client_eval_df is None:
            print("No client evaluation data found.")
            return

        metrics = ['accuracy', 'precision', 'recall', 'f1']
        titles = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
        
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        axes = axes.flatten()

        for i, metric in enumerate(metrics):
            if metric in self.client_eval_df.columns:
                sns.lineplot(data=self.client_eval_df, x='round', y=metric, 
                             hue='client-name', style='client-name', 
                             markers=True, ax=axes[i])
                
                axes[i].set_title(titles[i])
                axes[i].set_ylabel('Value')
                
                # Handle legend: Show only on the first plot to save space, 
                # or place outside. Let's place outside for the grid.
                if i == 0:
                    axes[i].legend(loc='center left', bbox_to_anchor=(1.05, 0.5), 
                                   framealpha=1.0, title='Client')
                else:
                    axes[i].get_legend().remove() if axes[i].get_legend() else None
        
        # Adjust layout to make room for the legend of the first plot
        plt.tight_layout(rect=[0, 0, 0.9, 1]) # Adjust rect to make space on right
        plt.show()

    def plot_all(self):
        """Runs all visualization methods."""
        print("\n" + "="*20 + " VISUALIZING EXPERIMENT " + "="*20)
        
        print("\n--- Server Metrics ---")
        self.plot_server_metrics()
        
        print("\n--- Client vs Server Comparison ---")
        self.plot_client_vs_server()
        
        print("\n--- System Performance ---")
        self.plot_system_performance()
        
        print("\n--- Straggler Analysis ---")
        self.plot_straggler_analysis()
        
        print("\n--- Client Metrics Grid ---")
        self.plot_client_metrics_grid()
        
        print("\n--- Global Confusion Matrix ---")
        self.plot_global_confusion_matrix()

scp -r host6:~/lxd-ovs-scripts/compressed_images_wheat/experiments compressed_images_wheat

In [ ]:
import glob

data_path = "compressed_images_wheat/experiments"
all_exps = glob.glob(f"{data_path}/*", recursive=True)
for x in all_exps:
    print(f"\"{x}\",")

In [ ]:
experiment_paths = [
"compressed_images_wheat/experiments/cifar10_mobilenet_v3_large_2026-04-30_10-30-40",
]

# for x in experiment_paths:
for x in all_exps:
    visualizer = ExperimentVisualizer(x)
    visualizer.plot_all()